# Feature Engineering

통계분석에서 확인한 주요 가격 결정 요인을 기반으로
머신러닝 모델에 사용할 입력 변수를 생성한다.

이번 프로젝트에서는 두 가지 예측 문제를 구성한다.

## Model A. 전세보증금 예측

Target:
- 전세보증금

주요 Feature:
- 자치구
- 법정동
- 전용면적
- 건축연식
- 층
- 계약연도
- 계약월

## Model B. 월세 예측

Target:
- 월세

주요 Feature:
- 보증금
- 자치구
- 법정동
- 전용면적
- 건축연식
- 층
- 계약연도
- 계약월

## Feature Engineering 원칙

- Target에서 직접 계산된 변수는 Feature로 사용하지 않는다.
- 전세보증금 예측 시 `deposit_per_m2`는 데이터 누수이므로 제외한다.
- `pyeong`과 `임대면적`은 동일 정보를 가지므로 기본 모델에서는 임대면적만 사용한다.
- 가격 분포의 왜도를 완화하기 위해 Target에는 로그 변환을 적용한다.
- 시간 정보를 고려하기 위해 계약연도와 계약월을 사용한다.
- 모델 평가 시 미래 데이터가 학습 데이터에 섞이지 않도록 시간 기준 분할을 사용한다.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

PROCESSED_DIR = Path("../data/processed")

file_path = (
    PROCESSED_DIR
    / "seoul_apartment_rental_2023_2025_clean.csv"
)

df = pd.read_csv(
    file_path,
    parse_dates=["contract_date"]
)

df.shape

(881794, 25)

In [2]:
df_model = df[
    (df["contract_date"] >= "2023-01-01")
    & (df["contract_date"] <= "2025-12-31")
].copy()

print("모델링 전체 데이터:", df_model.shape)
print()
print(df_model["전월세구분"].value_counts())

모델링 전체 데이터: (875925, 25)

전월세구분
전세    505485
월세    370440
Name: count, dtype: int64


In [3]:
df_model["contract_month_sin"] = np.sin(
    2 * np.pi * df_model["contract_month"] / 12
)

df_model["contract_month_cos"] = np.cos(
    2 * np.pi * df_model["contract_month"] / 12
)

df_model[
    [
        "contract_date",
        "contract_year",
        "contract_month",
        "contract_month_sin",
        "contract_month_cos"
    ]
].head()

,contract_date,contract_year,contract_month,contract_month_sin,contract_month_cos
0,2023-05-15,2023,5,0.5,-0.866025
1,2023-05-15,2023,5,0.5,-0.866025
2,2023-05-15,2023,5,0.5,-0.866025
3,2023-05-15,2023,5,0.5,-0.866025
4,2023-05-15,2023,5,0.5,-0.866025


In [9]:
base_year = 2023

df_model["time_idx"] = (
    (df_model["contract_year"] - base_year) * 12
    + (df_model["contract_month"] - 1)
)

df_model[
    [
        "contract_date",
        "contract_year",
        "contract_month",
        "time_idx"
    ]
].head()

,contract_date,contract_year,contract_month,time_idx
0,2023-05-15,2023,5,4
1,2023-05-15,2023,5,4
2,2023-05-15,2023,5,4
3,2023-05-15,2023,5,4
4,2023-05-15,2023,5,4


In [11]:
jeonse_features = [
    "자치구명",
    "법정동명",
    "임대면적",
    "building_age",
    "층",
    "time_idx",
    "contract_month_sin",
    "contract_month_cos"
]

jeonse_target = "보증금(만원)"

In [12]:
df_jeonse_model = df_model[
    (df_model["전월세구분"] == "전세")
    & (df_model["has_build_year"])
    & (df_model["보증금(만원)"] > 0)
].copy()

df_jeonse_model = df_jeonse_model[
    jeonse_features + [jeonse_target, "contract_date"]
].dropna()

df_jeonse_model["log_deposit"] = np.log1p(
    df_jeonse_model["보증금(만원)"]
)

print(
    "전세 모델 데이터:",
    df_jeonse_model.shape
)

df_jeonse_model.head()

전세 모델 데이터: (505418, 11)


,자치구명,법정동명,임대면적,building_age,층,time_idx,contract_month_sin,contract_month_cos,보증금(만원),contract_date,log_deposit
0,중랑구,묵동,84.71,20.0,3.0,4,0.5,-0.866025,46000,2023-05-15,10.736418
1,중랑구,묵동,101.97,20.0,4.0,4,0.5,-0.866025,46500,2023-05-15,10.747229
2,양천구,목동,133.03,19.0,5.0,4,0.5,-0.866025,70000,2023-05-15,11.156265
3,중랑구,망우동,84.94,17.0,14.0,4,0.5,-0.866025,37500,2023-05-15,10.532123
4,성동구,금호동2가,84.98,11.0,6.0,4,0.5,-0.866025,70000,2023-05-15,11.156265


## 전세 모델 Train/Test 분할

부동산 가격 예측은 과거 데이터를 이용해 미래 가격을 예측하는 상황을 가정한다.

따라서 데이터를 무작위로 섞지 않고 계약일을 기준으로 시간 순서대로 분리한다.

- Train: 2023~2024년 계약
- Test: 2025년 계약

이 방식을 통해 미래 데이터가 학습 과정에 포함되는 데이터 누수를 방지한다.

In [13]:
jeonse_train = df_jeonse_model[
    df_jeonse_model["contract_date"] < "2025-01-01"
].copy()

jeonse_test = df_jeonse_model[
    df_jeonse_model["contract_date"] >= "2025-01-01"
].copy()

print("Train:", jeonse_train.shape)
print("Test :", jeonse_test.shape)

print()
print("Train 기간:")
print(
    jeonse_train["contract_date"].min(),
    "~",
    jeonse_train["contract_date"].max()
)

print()
print("Test 기간:")
print(
    jeonse_test["contract_date"].min(),
    "~",
    jeonse_test["contract_date"].max()
)

Train: (266848, 11)
Test : (238570, 11)

Train 기간:
2023-01-01 00:00:00 ~ 2024-12-31 00:00:00

Test 기간:
2025-01-01 00:00:00 ~ 2025-12-31 00:00:00


In [7]:
print("Train 연도별:")
print(
    jeonse_train["contract_year"]
    .value_counts()
    .sort_index()
)

print()

print("Test 연도별:")
print(
    jeonse_test["contract_year"]
    .value_counts()
    .sort_index()
)

Train 연도별:
contract_year
2023    140748
2024    126100
Name: count, dtype: int64

Test 연도별:
contract_year
2025    238570
Name: count, dtype: int64


## 월세 예측 모델 Feature 구성

월세 예측에서는 보증금이 핵심 설명변수이므로
보증금을 입력 Feature에 포함한다.

보증금의 분포가 크게 치우쳐 있으므로
원값 대신 log(보증금)을 생성하여 사용한다.

Target인 월세 역시 오른쪽 꼬리가 긴 분포이므로
로그 변환된 월세를 모델 학습 Target으로 사용한다.

In [14]:
df_monthly_model = df_model[
    (df_model["전월세구분"] == "월세")
    & (df_model["valid_monthly_rent"])
    & (df_model["임대료(만원)"] > 0)
    & (df_model["보증금(만원)"] > 0)
    & (df_model["has_build_year"])
].copy()

df_monthly_model["log_deposit"] = np.log1p(
    df_monthly_model["보증금(만원)"]
)

df_monthly_model["log_rent"] = np.log1p(
    df_monthly_model["임대료(만원)"]
)

monthly_features = [
    "자치구명",
    "법정동명",
    "임대면적",
    "building_age",
    "층",
    "log_deposit",
    "time_idx",
    "contract_month_sin",
    "contract_month_cos"
]

df_monthly_model = df_monthly_model[
    monthly_features
    + [
        "임대료(만원)",
        "log_rent",
        "contract_date"
    ]
].dropna()

print(
    "월세 모델 데이터:",
    df_monthly_model.shape
)

월세 모델 데이터: (368881, 12)


## 월세 모델 Train/Test 분할

전세 모델과 동일하게 미래 예측 상황을 가정하여
계약일을 기준으로 데이터를 분할한다.

- Train: 2023~2024년 계약
- Test: 2025년 계약

Random Split을 사용하지 않고 시간 기준으로 분리하여
미래 데이터가 학습 과정에 포함되는 것을 방지한다.

In [15]:
monthly_train = df_monthly_model[
    df_monthly_model["contract_date"] < "2025-01-01"
].copy()

monthly_test = df_monthly_model[
    df_monthly_model["contract_date"] >= "2025-01-01"
].copy()

print("Monthly Train:", monthly_train.shape)
print("Monthly Test :", monthly_test.shape)

print()
print("Train 기간:")
print(
    monthly_train["contract_date"].min(),
    "~",
    monthly_train["contract_date"].max()
)

print()
print("Test 기간:")
print(
    monthly_test["contract_date"].min(),
    "~",
    monthly_test["contract_date"].max()
)

Monthly Train: (185103, 12)
Monthly Test : (183778, 12)

Train 기간:
2023-01-01 00:00:00 ~ 2024-12-31 00:00:00

Test 기간:
2025-01-01 00:00:00 ~ 2025-12-31 00:00:00


In [16]:
print("Train 연도별:")
print(
    monthly_train["contract_date"]
    .dt.year
    .value_counts()
    .sort_index()
)

print()

print("Test 연도별:")
print(
    monthly_test["contract_date"]
    .dt.year
    .value_counts()
    .sort_index()
)

Train 연도별:
contract_date
2023    94505
2024    90598
Name: count, dtype: int64

Test 연도별:
contract_date
2025    183778
Name: count, dtype: int64


## 최종 모델링 데이터 저장

Feature Engineering과 시간 기준 Train/Test 분할이 완료된 데이터를
다음 모델링 단계에서 바로 사용할 수 있도록 저장한다.

저장 데이터:

- 전세 Train: 2023~2024
- 전세 Test: 2025
- 월세 Train: 2023~2024
- 월세 Test: 2025

모델링 데이터는 원본 및 정제 데이터와 분리하여 관리한다.

In [17]:
MODEL_DATA_DIR = PROCESSED_DIR / "modeling"

MODEL_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

jeonse_train.to_csv(
    MODEL_DATA_DIR / "jeonse_train.csv",
    index=False,
    encoding="utf-8-sig"
)

jeonse_test.to_csv(
    MODEL_DATA_DIR / "jeonse_test.csv",
    index=False,
    encoding="utf-8-sig"
)

monthly_train.to_csv(
    MODEL_DATA_DIR / "monthly_train.csv",
    index=False,
    encoding="utf-8-sig"
)

monthly_test.to_csv(
    MODEL_DATA_DIR / "monthly_test.csv",
    index=False,
    encoding="utf-8-sig"
)

print("모델링 데이터 저장 완료")

모델링 데이터 저장 완료


In [18]:
for file in MODEL_DATA_DIR.glob("*.csv"):
    size_mb = file.stat().st_size / (1024 ** 2)

    print(
        file.name,
        f"{size_mb:.2f} MB"
    )

jeonse_test.csv 25.29 MB
jeonse_train.csv 28.11 MB
monthly_test.csv 22.08 MB
monthly_train.csv 22.07 MB


## Feature Engineering 결과

전세와 월세 예측을 위한 Feature Engineering을 완료하였다.

### 전세 모델

- Train: 266,848건
- Test: 238,570건
- Target: 전세보증금
- 모델 학습 Target: log(전세보증금)

주요 Feature:
- 자치구
- 법정동
- 전용면적
- 건축연식
- 층
- 시간 추세
- 계절성

### 월세 모델

- Train: 185,103건
- Test: 183,778건
- Target: 월세
- 모델 학습 Target: log(월세)

주요 Feature:
- 보증금
- 자치구
- 법정동
- 전용면적
- 건축연식
- 층
- 시간 추세
- 계절성

### 데이터 분할

미래 데이터 누수를 방지하기 위해 Random Split 대신
계약일을 기준으로 시간 순서대로 데이터를 분리하였다.

- Train: 2023~2024
- Test: 2025

다음 단계에서는 동일한 Train/Test 데이터셋을 사용하여
Baseline 모델과 머신러닝 모델의 예측 성능을 비교한다.